In [1]:
import duckdb
import polars as pl
import polars.selectors as cs

In [2]:
# get predictions from python and database
with duckdb.connect('../dev.duckdb') as con:
    df_pred_db = con.table("pred_churn").pl()

df_pred_py = pl.read_csv('preds_py.csv')

df_preds = df_pred_db.join(df_pred_py, on = 'customer_id', suffix = '_py')
df_preds.head()

pred,customer_id,model_version,pred_py
f64,str,str,f64
0.435064,"""7590-VHVEG""","""1.0""",0.435064
0.140688,"""5575-GNVDE""","""1.0""",0.140688
0.349945,"""3668-QPYBK""","""1.0""",0.349945
0.108988,"""7795-CFOCW""","""1.0""",0.108988
0.581118,"""9237-HQITU""","""1.0""",0.581118


In [3]:
# find any differences
(
df_preds
.filter( (pl.col('pred') - pl.col('pred_py')).abs() > 0.0001)
.select('customer_id', 'pred_py', pl.col('pred').alias('pred_db'))
.with_columns( cs.numeric().cast(pl.Float32).round(4) )
)

customer_id,pred_py,pred_db
str,f32,f32
"""4472-LVYGI""",0.1629,0.1658
"""3115-CZMZD""",0.0844,0.0915
"""5709-LVOEQ""",0.1712,0.1722
"""4367-NUYAO""",0.07,0.0775
"""9359-UGBTK""",0.1251,0.1272
…,…,…
"""2520-SGTTA""",0.0736,0.0812
"""2923-ARZLG""",0.1044,0.1147
"""9728-FTTVZ""",0.582,0.5839
